# Follow-up analyses: temporal overlap, BERT truncation, main table, hybrid model (E4.1), BERT at MAX_LEN=288

Run after the factorial (Point 4) and k-NN (Point 7) notebooks, whose JSON results must be on Drive. Cells run in order; only the E4.1 cells, E4.2 and the BERT@288 cells train models.

| Part | Content | Training? |
|---|---|---|
| Temporal overlap | Overlap between train and test episodes in recording time | no |
| E1.4 / E3.6 | BERT token length and truncation (MAX_LEN 128 vs 288) | no |
| D4 | Main table: macro-F1 and balanced accuracy (accuracy as complement) | no (reads JSON) |
| E4.2 | Latency | light |
| E4.4 | Coupling population check | no |
| E4.1 | Hybrid model, ablations, baselines on the same test keys; diagnostics T1–T5, V1–V3 | yes |
| BERT@288 | BERT cell D (optionally C) at MAX_LEN=288; consistent 2×2 | yes (GPU) |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, sys, os, json
import numpy as np, pandas as pd

P15 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'
P25 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'

def load_and_filter(path, target_speed):
    df = pd.read_csv(path)
    df['label'] = df['Scenario'].map({1: 0, 2: 1, 3: 2})
    df = df.sort_values(['Speed', 'Scenario', 'Time']).reset_index(drop=True)
    df['pseudo_run_id'] = (df['Speed'].round(2).astype(str) + '_' +
                           df['Scenario'].astype(str) + '_' +
                           (df['Time'] // 60).astype(int).astype(str))
    lo, hi = {15: (13, 17), 25: (23, 27)}[target_speed]
    return df[(df['Speed'] >= lo) & (df['Speed'] <= hi)].copy()

shutil.copy('/content/drive/MyDrive/Detector_Results/pcmef_fixed.py', '/content/pcmef_fixed.py')
if '/content' not in sys.path: sys.path.insert(0, '/content')
import pcmef_fixed as P

dfs15_raw, dfs25_raw = load_and_filter(P15, 15), load_and_filter(P25, 25)
dfs15, _, d15 = P.filter_feasible_episodes(dfs15_raw, min_duration_sec=15)
dfs25, _, d25 = P.filter_feasible_episodes(dfs25_raw, min_duration_sec=15)
print(f'15 m/s dibuang {d15} | 25 m/s dibuang {d25}  (harus 130 / 115)')
OUT = '/content/drive/MyDrive/Detector_Results/factorial'
OUT_NEXT = '/content/drive/MyDrive/Detector_Results/tahap_lanjutan'
os.makedirs(OUT_NEXT, exist_ok=True)

## Temporal overlap between episodes

**Question:** does the split by `pseudo_run_id` (rounded Speed + Scenario + 60 s bucket) actually separate *time*, or do test and train windows come from the same seconds of the same recording?

**Input:** `dfs15_raw`/`dfs25_raw` (all rows) and `dfs15`/`dfs25` (39-episode population) from the setup cell. No model training.

**Output:** concurrency per Scenario, overlap distribution per seed (row and window level), Speed–Time figure. Saved to `OUT_NEXT`. Cells 1.1–1.3 describe the data; cell 1.4 gives the overlap figures reported in the manuscript.

In [ ]:
# 1.1 Arti Speed & Time: berapa (Scenario, Time) yang punya >1 baris? (>1 = beberapa node/kendaraan direkam bersamaan)
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
for sp, raw in ((15, dfs15_raw), (25, dfs25_raw)):
    dup = raw.groupby(['Scenario', 'Time']).size()
    print(f'{sp} m/s: fraksi (Scenario,Time) dgn >1 baris = {(dup>1).mean():.3f} | maks baris pada Time yg sama = {dup.max()}')
    print('   Speed unik per Scenario:', raw.groupby('Scenario')['Speed'].nunique().to_dict())
    fig, axs = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
    for ax, (scn, g) in zip(axs, raw.groupby('Scenario')):
        ax.scatter(g['Time'], g['Speed'], s=3, alpha=.5)
        ax.set_title(f'{sp} m/s - Scenario {scn}'); ax.set_xlabel('Time (s)')
    axs[0].set_ylabel('Speed'); plt.tight_layout()
    plt.savefig(f'{OUT_NEXT}/L1_speed_vs_time_{sp}ms.png', dpi=120); plt.close()
print('Gambar Speed-Time disimpan di', OUT_NEXT)
print('Baca: banyak jalur Speed paralel -> Speed membedakan node; satu kurva -> kecepatan satu kendaraan.')
print('Cocokkan dengan dokumentasi Kosmanos [22]; kalau tidak dijelaskan, tulis apa yang terlihat + "interpretasi belum terverifikasi".')

In [ ]:
# 1.2 Profil episode + 1.3 konkurensi per detik (populasi 39 episode)
prof_all, conc_all = [], []
for sp, d in ((15, dfs15), (25, dfs25)):
    ep = d.groupby('pseudo_run_id').agg(scenario=('Scenario','first'), t_min=('Time','min'),
                                         t_max=('Time','max'), n=('Time','size'))
    ep['span'] = ep.t_max - ep.t_min
    ep['rows_per_s'] = ep.n / ep.span.clip(lower=1e-9)
    tot = d.groupby('Scenario').agg(n=('Time','size'), t_min=('Time','min'), t_max=('Time','max'))
    tot['rows_per_s_total'] = tot.n / (tot.t_max - tot.t_min).clip(lower=1e-9)
    s = ep.groupby('scenario')['rows_per_s'].median().rename('median_rows_per_s_episode').to_frame()
    s = s.join(tot['rows_per_s_total']); s['rasio_total/episode'] = s.rows_per_s_total / s.median_rows_per_s_episode
    print(f'\n=== {sp} m/s: laju baris per episode vs total (rasio >>1 = episode berjalan bersamaan) ==='); print(s.round(2))
    d = d.copy(); d['sec'] = d['Time'].astype(int)
    conc = d.groupby(['Scenario','sec'])['pseudo_run_id'].nunique()
    cs = conc.groupby('Scenario').describe(); cs['frac_detik_>1_episode'] = conc.gt(1).groupby('Scenario').mean()
    print(f'--- {sp} m/s: jumlah episode aktif per detik (>1 = tumpang tindih langsung) ---'); print(cs.round(3))
    prof_all.append(s.assign(speed=sp)); conc_all.append(cs.assign(speed=sp))
pd.concat(prof_all).to_csv(f'{OUT_NEXT}/L1_profil_episode.csv'); pd.concat(conc_all).to_csv(f'{OUT_NEXT}/L1_konkurensi.csv')

In [ ]:
# 1.4 Overlap train vs test per seed (populasi 39 episode): level BARIS (detik yg sama) & level WINDOW (interval beririsan)
def overlap_one(d, seed):
    tr, va, te = P.split_by_episode(d, seed=seed)
    key = lambda x: set(zip(x.Scenario, x['Time'].astype(int)))
    k_tr, k_te = key(tr), key(te)
    row_ov = len(k_te & k_tr) / max(1, len(k_te))
    sc_of = d.groupby('pseudo_run_id')['Scenario'].first().to_dict()
    _, _, ktr = P.create_windows_keyed(tr, 15, 1); _, _, kte = P.create_windows_keyed(te, 15, 1)
    starts = {}
    for rid, t0 in ktr: starts.setdefault(sc_of[rid], []).append(t0)
    starts = {k: np.sort(v) for k, v in starts.items()}
    hit = 0
    for rid, t0 in kte:
        u = starts.get(sc_of[rid]);  # window train [u,u+15) beririsan dgn test [t0,t0+15) <=> t0-15 < u < t0+15
        if u is not None and np.searchsorted(u, t0 + 15, 'left') - np.searchsorted(u, t0 - 15, 'right') > 0: hit += 1
    return dict(seed=seed, n_test_win=len(kte), overlap_baris=row_ov, overlap_window=hit / max(1, len(kte)))

ov = {}
for sp, d in ((15, dfs15), (25, dfs25)):
    ov[sp] = pd.DataFrame([overlap_one(d, s) for s in P.SEEDS])
    ov[sp].to_csv(f'{OUT_NEXT}/L1_overlap_per_seed_{sp}ms.csv', index=False)
    print(f'\n=== {sp} m/s, 20 seed: overlap test<->train ===')
    print((ov[sp][['overlap_baris','overlap_window']]*100).describe().round(2))
    m = ov[sp].overlap_window.mean() * 100
    g2 = ('Overlap ~0%: episode terpisah dlm waktu -> pertahankan framing PCMEF' if m < 1 else
          'Overlap sebagian (1-50%): framing PCMEF dilunakkan; laporkan angkanya' if m <= 50 else
          'Overlap TINGGI (>50%): split per episode praktis tidak memisahkan waktu -> framing berubah (lihat 1.6)')
    print(f'>> G2 ({sp} m/s): overlap window rata-rata = {m:.1f}% -> {g2}')
print('\nThe [X]% in the manuscript draft is the mean window overlap.')

## E1.4 / E3.6: BERT token length and truncation

Tokenises 15 s windows (source and modified schemes) and reports mean, p95, p99 and the share of truncated texts at MAX_LEN=128 (modified), 288 (modified) and the dynamic MAX_LEN (source). Measurement only.

In [ ]:
from transformers import BertTokenizerFast, DistilBertTokenizerFast
tok_src = BertTokenizerFast.from_pretrained('bert-base-uncased')
tok_mod = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def token_report(df, speed):
    X_l, y, keys = P.create_windows_keyed(df, 15, 1)
    fs = P.compute_feat_stats(df)   # utk panjang token, bin dari seluruh populasi cukup (bukan utk evaluasi)
    t_src = [P._tokens_source(x) for x in X_l]
    t_mod = [P._tokens_modified(x, fs, n_bins=20) for x in X_l]
    L_src = np.array([len(tok_src.encode(t)) for t in t_src])
    L_mod = np.array([len(tok_mod.encode(t)) for t in t_mod])
    # MAX_LEN dinamis source = min(512, max panjang token 200 teks train pertama) -- ditiru dari run_bert_cell
    dyn = int(min(512, L_src[:200].max()))
    def stats(L, cap):
        return dict(n=int(len(L)), mean=float(L.mean()), p95=float(np.percentile(L,95)),
                    p99=float(np.percentile(L,99)), max=int(L.max()),
                    max_len=int(cap), pct_truncated=float((L > cap).mean()*100),
                    mean_frac_kept=float(np.minimum(1, cap/L).mean()))
    rep = dict(speed=speed,
               modified=stats(L_mod, 128),
               modified_288=stats(L_mod, 288),
               source_dynamic=stats(L_src, dyn),
               source_at_512=stats(L_src, 512))
    # per kelas utk modified
    rep['modified_pct_truncated_per_class'] = {int(c): float((L_mod[y==c] > 128).mean()*100) for c in np.unique(y)}
    return rep

reports = [token_report(dfs15, 15), token_report(dfs25, 25)]
for r in reports:
    print(f"\n=== {r['speed']} m/s ===")
    for k in ('modified','modified_288','source_dynamic','source_at_512'):
        s = r[k]
        print(f"  {k:15s} MAX_LEN={s['max_len']:3d} | mean={s['mean']:.1f} p95={s['p95']:.0f} p99={s['p99']:.0f} "
              f"max={s['max']} | terpotong={s['pct_truncated']:.2f}% | rata2 isi window yg terbaca={s['mean_frac_kept']*100:.1f}%")
    print('  modified terpotong per kelas (%):', r['modified_pct_truncated_per_class'])
json.dump(reports, open(f'{OUT_NEXT}/E1_4_token_length.json','w'), indent=2)

## D4: main table

Reads every cell JSON (RF-Stat, LSTM, BERT × 15/25 m/s × A/B/C/D × 20 seeds) from `factorial`. Macro-F1 and balanced accuracy are primary; accuracy is complementary. Only files whose `code_hash` matches the current code are used; stale files are skipped and flagged.

In [ ]:
import glob
rows, stale = [], 0
for fp in glob.glob(f'{OUT}/*.json'):
    try: r = json.load(open(fp))
    except Exception: continue
    mname = os.path.basename(fp).split('_1')[0].split('_2')[0]   # RF_Stat / LSTM / BERT / KNN
    mname = {'RF':'RF_Stat','KNN':'KNN'}.get(mname, mname)
    if mname.startswith('RF'): mname = 'RF_Stat'
    if mname not in P._MODEL_FUNCS or not P.cache_is_valid(r, mname):
        stale += 1; continue
    if 'cell' not in r or 'accuracy' not in r: continue
    sp = 15 if '_15ms_' in fp else 25 if '_25ms_' in fp else None
    rows.append(dict(model=r.get('model'), speed=sp, cell=r['cell'], seed=r['seed'],
                     acc=r['accuracy']*100, f1=r.get('f1_macro', np.nan)*100,
                     bacc=r.get('balanced_accuracy', np.nan)*100,
                     maj=r.get('majority_class_baseline_accuracy', np.nan)*100))
print(f'{len(rows)} file valid | {stale} file basi (code_hash beda) diabaikan')
df_all = pd.DataFrame(rows)
def ms(x): return f'{x.mean():.2f} ± {x.std(ddof=1):.2f}'
tab = (df_all.groupby(['model','speed','cell'])
       .agg(n=('seed','nunique'), macroF1=('f1',ms), bal_acc=('bacc',ms), acc=('acc',ms), maj_base=('maj','mean'))
       .reset_index())
tab['maj_base'] = tab['maj_base'].round(1)
display(tab)
tab.to_csv(f'{OUT_NEXT}/D4_tabel_utama.csv', index=False)
print('Cek: setiap baris harus n=20. Baris n<20 = ada seed hilang/basi -> jalankan ulang notebook asalnya.')

## E4.2: latency

Re-measures only configurations whose preprocessing or architecture changed (LSTM source: gated architecture; BERT if MAX_LEN changes). Median ms per window (batch=1), 200 runs after 20 warm-ups; random weights are enough for inference latency.

In [ ]:
import time, tensorflow as tf
def bench(fn, n=200, warm=20):
    for _ in range(warm): fn()
    t=[]
    for _ in range(n):
        s=time.perf_counter(); fn(); t.append((time.perf_counter()-s)*1000)
    return float(np.median(t)), float(np.percentile(t,95))

X_l, y, _ = P.create_windows_keyed(dfs15, 15, 1)
Xp = P.pad_list(X_l)[0] if isinstance(P.pad_list(X_l), tuple) else P.pad_list(X_l)
xb = Xp[:1].astype('float32')
res = {}
for arch in ('source','modified'):
    m = P.build_lstm(arch, Xp.shape[1:], seed=0)
    f = tf.function(lambda x: m(x, training=False))      # tanpa overhead predict() Keras
    xt = tf.constant(xb)
    res[f'LSTM_{arch}'] = bench(lambda: f(xt).numpy())
for k,v in res.items(): print(f'{k:16s} median={v[0]:.2f} ms  p95={v[1]:.2f} ms  (batch=1, tf.function, device={"GPU" if tf.config.list_physical_devices("GPU") else "CPU"})')
json.dump(res, open(f'{OUT_NEXT}/E4_2_latency_lstm.json','w'), indent=2)
print('\nBERT latency is measured separately if MAX_LEN changes.')

## E4.4: coupling (Tables 8–9, Figure 5), population check

If coupling was computed on raw samples per regime, no recomputation is needed; if it used the filtered population, it must be recomputed. This cell shows how much the population changes.

In [ ]:
for sp, raw, fil in ((15, dfs15_raw, dfs15), (25, dfs25_raw, dfs25)):
    print(f'{sp} m/s: baris {len(raw)} -> {len(fil)} | episode {raw.pseudo_run_id.nunique()} -> {fil.pseudo_run_id.nunique()}')
print('''
Coupling computed on all raw samples per regime (no episode filter): no recomputation needed.
Coupling computed on the same filtered windows/episodes as Point 4: recompute with dfs15/dfs25 above (39-episode population).
''')

## E4.4b: coupling on the 39-episode population (comparison)

The original Tables 8–9 (`t10_feature_coupling`) use raw samples per regime (n=1195 @25 m/s, n=1172 @15 m/s). This cell adds the filtered 39-episode version with the same function (Pearson, Spearman, MI/H(Y) in %).

**Validity check:** the `*_mentah` (raw) columns must equal Tables 8–9, e.g. 25 m/s: SNR r=0.279, MI=74.7%; 15 m/s: Relative_Speed r=-0.576, MI=55.2%.

In [ ]:
from scipy.stats import pearsonr, spearmanr, entropy
from sklearn.feature_selection import mutual_info_classif

CLASS_NAMES = ['Interference', 'Reactive', 'Constant']   # label 0,1,2 (sama dgn common.py / t10)

def norm_mi(x, y, seed=0):
    mi = mutual_info_classif(x.reshape(-1, 1), y, discrete_features=False, random_state=seed)[0]
    _, cnt = np.unique(y, return_counts=True)
    Hy = entropy(cnt)
    return 100.0 * mi / Hy if Hy > 0 else 0.0

def coupling_df(df):
    y = df['label'].values
    rows = []
    for f in P.FEATURES:
        x = df[f].values.astype(float)
        ok = np.std(x) > 0
        rows.append(dict(feature=f,
                         pearson=round(float(pearsonr(x, y)[0]) if ok else 0.0, 3),
                         spearman=round(float(spearmanr(x, y)[0]) if ok else 0.0, 3),
                         mi_pct=round(float(norm_mi(x, y)), 1)))
    rs = {CLASS_NAMES[c]: round(float(df.loc[df['label'] == c, 'Relative_Speed'].mean()), 2) for c in range(3)}
    return pd.DataFrame(rows).set_index('feature'), rs

coup_all = []
for sp, raw, fil in ((25, dfs25_raw, dfs25), (15, dfs15_raw, dfs15)):
    t_raw, rs_raw = coupling_df(raw)
    t_fil, rs_fil = coupling_df(fil)
    comp = t_raw.join(t_fil, lsuffix='_mentah', rsuffix='_39ep')
    comp['dMI_pp'] = (comp['mi_pct_39ep'] - comp['mi_pct_mentah']).round(1)
    print(f"\n=== Coupling @ {sp} m/s | mentah n={len(raw)} vs terfilter n={len(fil)} (39 episode) ===")
    display(comp)
    print('  Relative_Speed rata-rata per kelas, mentah :', rs_raw)
    print('  Relative_Speed rata-rata per kelas, 39 ep. :', rs_fil)
    c = comp.reset_index(); c.insert(0, 'speed', sp); coup_all.append(c)
pd.concat(coup_all).to_csv(f'{OUT_NEXT}/E4_4_coupling_mentah_vs_39ep.csv', index=False)
print('\nTersimpan: E4_4_coupling_mentah_vs_39ep.csv')


## E4.1: hybrid model, ablations and baselines on the same population and test keys (20 seeds)

**Input:** `dfs15` (39 episodes after `filter_feasible_episodes(min_duration_sec=15)`) and the latest `pcmef_fixed.py` (`e41_*` block).

**Per seed:** three BiLSTMs (1/5/15 s) + tabular XGBoost + SpatialKNN, then a meta-XGBoost trained on **validation**. Four meta configurations (full, no KNN, no XGB, embeddings only), four single models (BiLSTM 1/5/15 s, XGB-only), and the RF-Stat D / LSTM D baselines. All are scored on the **same test keys K** (n_test asserted identical).

**Output:** JSON + NPZ with per-window predictions in `OUT_E41`; Tables 2, 4, 5, 6; Figures 3–4.

Notes: the meta-learner is trained on validation; same-episode neighbours are excluded in SpatialKNN; scaling is fit before padding, and the BiLSTMs have no Masking layer.

In [ ]:
# 2.2 Jalankan E4.1 (resumable: seed yang sudah ada dipakai ulang). ~3 BiLSTM + XGB + baseline per seed; mulai dgn 1-2 seed utk uji waktu.
OUT_E41 = '/content/drive/MyDrive/Detector_Results/e41_hybrid'
os.makedirs(OUT_E41, exist_ok=True)
E41_SEEDS = P.SEEDS            # 20 seed; ganti P.SEEDS[:2] utk uji coba
E41_BASELINES_RUN = ('RF_Stat', 'LSTM')   # tambah 'BERT' kalau GPU tersedia & waktu cukup (sel D, MAX_LEN=128)
res41 = P.e41_run_all(dfs15, 15, OUT_E41, seeds=E41_SEEDS, baselines=E41_BASELINES_RUN)
print(f'\nselesai: {len(res41)} seed')

In [ ]:
# 2.3 Agregasi: Tabel 2 (hybrid), Tabel 4 (hybrid vs baseline), Tabel 5 (ablasi), Tabel 6 (importance blok) + sanity check
import glob
R41 = [json.load(open(f)) for f in sorted(glob.glob(f'{OUT_E41}/E41_15ms_seed*.json'))]
R41 = [r for r in R41 if r.get('e41_version') == P.E41_VERSION]
print('seed tersedia:', len(R41))
vals = {}
for r in R41:
    for cfg, m in r['metrics'].items(): vals.setdefault(cfg, {})[r['seed']] = m

def ms(cfg, k):
    a = np.array([vals[cfg][s][k] for s in sorted(vals[cfg])]) * 100
    return a.mean(), a.std(ddof=1) if len(a) > 1 else float('nan')

# --- Tabel 2: hybrid (mean±SD, 3 metrik)
print('\n=== Tabel 2 — hybrid_full (n seed = %d) ===' % len(vals['hybrid_full']))
t2 = pd.DataFrame({k: '%.2f ± %.2f' % ms('hybrid_full', k) for k in ('accuracy', 'f1_macro', 'balanced_accuracy')}, index=['hybrid_full'])
print(t2.to_string())

# --- Tabel 4: hybrid vs baseline (berpasangan per seed, n_test identik)
base = [c for c in ('RF_Stat_D', 'LSTM_D', 'BERT_D') if c in vals]
print('\n=== Tabel 4 — hybrid_full vs baseline sel D (selisih = hybrid − baseline, pp; Holm antar baseline) ===')
t4 = {}
for met in ('accuracy', 'f1_macro', 'balanced_accuracy'):
    t4[met] = P.e41_paired_table(vals, 'hybrid_full', base, metric=met)
    print(f'\n[{met}]'); print(t4[met].round(3).to_string(index=False))

# --- Tabel 5: ablasi (tiap konfigurasi vs hybrid_full)
abl = [c for c in ('multi_no_knn', 'multi_no_xgb', 'multi_emb_only', 'bilstm_1s', 'bilstm_5s', 'bilstm_15s', 'xgb_only') if c in vals]
print('\n=== Tabel 5 — ablasi (selisih = hybrid_full − konfigurasi, pp) ===')
rows = [dict(config=c, **{k: '%.2f ± %.2f' % ms(c, k) for k in ('accuracy', 'f1_macro', 'balanced_accuracy')}) for c in ['hybrid_full'] + abl]
print(pd.DataFrame(rows).to_string(index=False))
t5 = P.e41_paired_table(vals, 'hybrid_full', abl, metric='accuracy'); print('\n'); print(t5.round(3).to_string(index=False))

# --- Tabel 6: importance per blok (rata-rata seed, hybrid_full)
bi = pd.DataFrame([r['block_importance']['hybrid_full'] for r in R41])
print('\n=== Tabel 6 — importance blok meta-learner (hybrid_full), mean ± SD antar seed (%) ===')
t6 = pd.DataFrame({'mean%': bi.mean()*100, 'SD%': bi.std(ddof=1)*100}); print(t6.round(2).to_string())

# --- Sanity check wajib
print('\n=== Sanity check ===')
nv = [r['n_val_aligned'] for r in R41]; nt = [r['n_test_common'] for r in R41]; nat = [r['n_test_native'] for r in R41]
print(f'n_val aligned: min {min(nv)} / median {int(np.median(nv))} / max {max(nv)} | n_test: {min(nt)}–{max(nt)}')
print('seed dengan n_val < 60 (meta tanpa early-stopping internal):', [r['seed'] for r in R41 if not r['meta_info']['hybrid_full']['internal_split']])
print('seed dengan n_test_common != n_test_native (align_multi membuang window):', [(r['seed'], r['n_test_native'], r['n_test_common']) for r in R41 if r['n_test_common'] != r['n_test_native']])
gap = ms('hybrid_full', 'accuracy')[0] - ms('bilstm_15s', 'accuracy')[0]
print(f'hybrid − bilstm_15s = {gap:+.2f} pp', '(>2 pp: dugaan "hybrid unggul" didukung)' if gap > 2 else '(<=2 pp: JANGAN klaim hybrid unggul; laporkan apa adanya)')
for k in ('hybrid_full',):
    nocls = [(r['seed'], r['meta_info'][k]['val_class_counts']) for r in R41 if min(r['meta_info'][k]['val_class_counts']) < 4]
    print('seed dgn kelas val < 4 window:', nocls)
pd.concat({'T4_' + m: v for m, v in t4.items()}).to_csv(f'{OUT_NEXT}/L2_tabel4.csv')
t5.to_csv(f'{OUT_NEXT}/L2_tabel5.csv'); t6.to_csv(f'{OUT_NEXT}/L2_tabel6.csv'); t2.to_csv(f'{OUT_NEXT}/L2_tabel2.csv')

In [ ]:
# 2.4 (EKSPLORATIF) McNemar per-window, digabung antar seed — window stride-1 saling bergantung, jadi p-value TIDAK dipakai sbg bukti utama.
from scipy.stats import chi2
def mcnemar_pooled(cfg_a, cfg_b):
    b = c = 0
    for r in R41:
        z = np.load(f"{OUT_E41}/E41_15ms_seed{r['seed']}.npz")
        if f'pred_{cfg_a}' not in z.files or f'pred_{cfg_b}' not in z.files: continue
        ca, cb = z[f'pred_{cfg_a}'] == z['y_true'], z[f'pred_{cfg_b}'] == z['y_true']
        b += int((ca & ~cb).sum()); c += int((~ca & cb).sum())
    stat = (abs(b - c) - 1) ** 2 / max(1, b + c)
    return b, c, stat, 1 - chi2.cdf(stat, 1)
for o in base + ['bilstm_15s']:
    if o in vals:
        b, c, st, p = mcnemar_pooled('hybrid_full', o)
        print(f'hybrid_full vs {o:12s}: hanya-hybrid-benar={b:6d} hanya-lawan-benar={c:6d} chi2={st:9.1f} p={p:.2g}  (eksploratif)')

In [ ]:
# 2.5 Figure 3 (konvergensi antar-seed) & Figure 4 (importance blok hybrid_full + confusion matrix gabungan)
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
seeds_sorted = sorted(vals['hybrid_full'])
fig, ax = plt.subplots(figsize=(7, 4))
for cfg in ['hybrid_full'] + base + ['bilstm_15s']:
    if cfg not in vals: continue
    a = np.array([vals[cfg][s]['accuracy'] for s in seeds_sorted]) * 100
    ax.plot(range(1, len(a) + 1), np.cumsum(a) / np.arange(1, len(a) + 1), marker='o', ms=3, label=cfg)
ax.set_xlabel('jumlah seed'); ax.set_ylabel('akurasi kumulatif rata-rata (%)'); ax.legend(fontsize=7); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_NEXT}/L2_figure3_seed_convergence.png', dpi=150); plt.close()

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
t6['mean%'].plot.barh(xerr=t6['SD%'], ax=axs[0]); axs[0].set_xlabel('importance (%)'); axs[0].set_title('Blok meta-learner (hybrid_full)')
cm = np.sum([np.array(r['metrics']['hybrid_full']['confusion_matrix']) for r in R41], axis=0)
cmn = cm / cm.sum(1, keepdims=True)
im = axs[1].imshow(cmn, vmin=0, vmax=1, cmap='Blues'); axs[1].set_title('Confusion matrix (baris-ternormalisasi, semua seed)')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]): axs[1].text(j, i, f'{cmn[i, j]:.2f}', ha='center', va='center')
axs[1].set_xlabel('prediksi'); axs[1].set_ylabel('sebenarnya'); plt.tight_layout()
plt.savefig(f'{OUT_NEXT}/L2_figure4_importance_confusion.png', dpi=150); plt.close()
print('Figures 3 and 4 saved to', OUT_NEXT)

## Diagnostics for the hybrid comparison (T1–T5)

All without retraining. They need the setup cell (`P`, `dfs15`, `OUT`, `OUT_NEXT`) and the E4.1 cells (`OUT_E41`).

| Cell | Purpose |
|---|---|
| T1 | Why n_test drops in E4.1 (empty 1 s / 5 s windows) |
| T2 | Baselines on the native vs common-key subset population |
| T3 | Are per-window BERT D predictions stored? |
| T5 | Paired TOST (±2 pp margin, fixed before looking at results) for "not distinguishable" claims |

In [ ]:
# T1. Kenapa n_test turun? Hitung, per seed, berapa window 15 s test yang kehilangan pasangan di skala 1 s / 5 s (TANPA training).
# Kunci window = (pseudo_run_id, t0) dgn grid t0 yg sama di semua skala -> kunci hilang HANYA kalau window pendeknya tidak punya baris (kosong).
CLASS_NAMES = ['Interference', 'Reactive', 'Constant']   # label 0,1,2
rows, per_cls, dens = [], [], []
for seed in P.SEEDS:
    _, _, te = P.split_by_episode(dfs15, seed=seed)
    X15, y15, k15 = P.create_windows_keyed(te, 15, 1)
    _, _, k1 = P.create_windows_keyed(te, 1, 1)
    _, _, k5 = P.create_windows_keyed(te, 5, 1)
    s1, s5 = set(k1), set(k5)
    has1 = np.array([k in s1 for k in k15]); has5 = np.array([k in s5 for k in k15])
    keep = has1 & has5
    rows.append(dict(seed=seed, n15=len(k15), miss1=int((~has1).sum()), miss5=int((~has5).sum()), n_common=int(keep.sum()),
                     pct_drop=100 * (1 - keep.mean())))
    for c in (0, 1, 2):
        m = y15 == c
        per_cls.append(dict(seed=seed, kelas=c, n15=int(m.sum()), n_common=int((m & keep).sum())))
    nrow = np.array([len(x) for x in X15])
    dens.append(dict(seed=seed, rows_kept=nrow[keep].mean() if keep.any() else np.nan, rows_dropped=nrow[~keep].mean() if (~keep).any() else np.nan))
d1 = pd.DataFrame(rows); print(d1.to_string(index=False))
print(f"\nrata-rata window 15 s tanpa pasangan 1 s: {d1.miss1.sum()}/{d1.n15.sum()} ({100*d1.miss1.sum()/d1.n15.sum():.1f}%) | tanpa pasangan 5 s: {d1.miss5.sum()} ({100*d1.miss5.sum()/d1.n15.sum():.1f}%)")
print(f"rata-rata dibuang = {d1.pct_drop.mean():.1f}%  (cocokkan dgn n_test_common/native di Langkah 2)")
pc = pd.DataFrame(per_cls).groupby('kelas')[['n15', 'n_common']].sum(); pc['pct_tersisa'] = 100 * pc.n_common / pc.n15
pc.index = [f'{i} {CLASS_NAMES[i]}' for i in pc.index]; print('\nPer kelas (jumlah semua seed): apakah proporsi tersisa berbeda antar-kelas?'); print(pc.round(1))
print('\nKomposisi kelas: native ', (pc.n15 / pc.n15.sum()).round(3).to_dict(), '| subset ', (pc.n_common / pc.n_common.sum()).round(3).to_dict())
dd = pd.DataFrame(dens); print(f"\nBaris rata-rata per window 15 s: yg dipertahankan {dd.rows_kept.mean():.1f} vs yg dibuang {dd.rows_dropped.mean():.1f}  (dibuang lebih jarang = subset bias ke sampling rapat)")
d1.to_csv(f'{OUT_NEXT}/T1_penyebab_ntest.csv', index=False); pc.to_csv(f'{OUT_NEXT}/T1_per_kelas.csv')
print('If miss1 >> miss5 ~ 0, the cause is empty 1 s windows (no rows in that second).')

In [ ]:
# T2. Baselines on two populations: native (all 15 s test windows) vs common-key subset K.
# Rule fixed before looking at the results: |mean(native) - mean(subset)| < 2 pp -> subset used, both populations reported.
import glob
def load_cfg(prefix, cell, seed):
    fp = f'{OUT}/{prefix}_15ms_{cell}_seed{seed}.json'
    return json.load(open(fp)) if os.path.exists(fp) else None
R41 = [json.load(open(f)) for f in sorted(glob.glob(f'{OUT_E41}/E41_15ms_seed*.json'))]
rows = []
for model, cfg in (('RF_Stat', 'RF_Stat_D'), ('LSTM', 'LSTM_D')):
    for r in R41:
        nat = load_cfg(model, 'D', r['seed'])
        if nat is None or cfg not in r['metrics']: continue
        rows.append(dict(model=model, seed=r['seed'], n_native=nat['n_test'], n_native_e41=r['n_test_native'], n_subset=r['n_test_common'],
                         acc_native=nat['accuracy']*100, acc_subset=r['metrics'][cfg]['accuracy']*100,
                         f1_native=nat['f1_macro']*100, f1_subset=r['metrics'][cfg]['f1_macro']*100))
t2 = pd.DataFrame(rows)
print('sanity: n_test native (factorial) == n_test_native (E41)?', bool((t2.n_native == t2.n_native_e41).all()), '(kalau False: populasi native beda, jangan bandingkan)')
s = t2.groupby('model').agg(n_seed=('seed', 'size'), acc_native=('acc_native', 'mean'), acc_subset=('acc_subset', 'mean'),
                            f1_native=('f1_native', 'mean'), f1_subset=('f1_subset', 'mean'))
s['selisih_acc_pp'] = s.acc_subset - s.acc_native; s['selisih_f1_pp'] = s.f1_subset - s.f1_native
print(s.round(2).to_string())
for m_, g in t2.groupby('model'):
    dif = (g.acc_subset - g.acc_native).values
    print(f"{m_}: selisih per seed (subset-native) mean {dif.mean():+.2f} pp, SD {dif.std(ddof=1):.2f}, min {dif.min():+.2f}, maks {dif.max():+.2f}")
big = (s.selisih_acc_pp.abs() >= 2).any()
print('>> Rule |native - subset| < 2 pp:', 'NOT met' if big else 'met')
t2.to_csv(f'{OUT_NEXT}/T2_baseline_native_vs_subset.csv', index=False)

In [ ]:
# T3. BERT D: apakah prediksi per window tersimpan di JSON C-4? (kalau tidak -> BERT dikeluarkan dari Tabel 4, rujuk Tabel 3 + dekomposisi)
fp = f'{OUT}/BERT_15ms_D_seed42.json'
if os.path.exists(fp):
    j = json.load(open(fp)); print('kunci JSON BERT D:', sorted(j.keys()))
    has_pred = any(k in j for k in ('y_pred', 'y_true', 'predictions', 'pred'))
    print('>> prediksi per window tersimpan:', has_pred)
    print('   ' + ('predictions available: use them on key set K without retraining' if has_pred else
                    'NOT stored: BERT is left out of the hybrid-vs-baseline table and is compared in the main table and the decomposition'))
else:
    print('file tidak ditemukan:', fp)

In [ ]:
# T5. Paired TOST (equivalence), hybrid_full vs each comparator. MARGIN = +-2 pp, fixed BEFORE looking at the results; do not change it after seeing p-values.
# H0: |difference| >= margin; H1: |difference| < margin. p_TOST = max(p_lower, p_upper); equivalent if p_TOST < 0.05 (same as the 90% CI inside +-margin). Unit = seed (n=20, paired).
from scipy import stats as sps
MARGIN = 2.0
def tost_paired(a, b, margin=MARGIN):
    d = np.asarray(a) - np.asarray(b); n = len(d); m = d.mean(); se = d.std(ddof=1) / np.sqrt(n)
    p_lo = sps.t.sf((m + margin) / se, n - 1)      # H0: d <= -margin
    p_hi = sps.t.sf((margin - m) / se, n - 1)      # H0: d >=  margin
    tc = sps.t.ppf(0.95, n - 1)
    return dict(n=n, diff_pp=m, ci90_lo=m - tc * se, ci90_hi=m + tc * se, p_tost=max(p_lo, p_hi))
vals = {}
for r in R41:
    for cfg, mt in r['metrics'].items(): vals.setdefault(cfg, {})[r['seed']] = mt
others = [c for c in ('RF_Stat_D', 'LSTM_D', 'bilstm_15s', 'xgb_only', 'multi_no_knn', 'multi_no_xgb', 'multi_emb_only') if c in vals]
out = []
for met in ('f1_macro', 'balanced_accuracy', 'accuracy'):
    for o in others:
        sd = sorted(set(vals['hybrid_full']) & set(vals[o]))
        a = np.array([vals['hybrid_full'][s][met] for s in sd]) * 100; b = np.array([vals[o][s][met] for s in sd]) * 100
        out.append(dict(metric=met, vs=o, **tost_paired(a, b)))
tt = pd.DataFrame(out)
tt['p_holm'] = np.nan
for met in tt.metric.unique():
    idx = tt.index[tt.metric == met]; tt.loc[idx, 'p_holm'] = P.holm_bonferroni(tt.loc[idx, 'p_tost'].values)
tt['ekuivalen_±2pp'] = tt.p_holm < 0.05
print(f'TOST margin ±{MARGIN} pp (Holm per metrik)'); print(tt.round(3).to_string(index=False))
tt.to_csv(f'{OUT_NEXT}/T5_tost.csv', index=False)
print('\nBaca: ekuivalen=True -> boleh tulis "equivalent within ±2 pp" untuk pasangan itu. False -> tulis "no detectable difference" + catatan daya uji rendah (unit efektif 6 episode per seed).')

In [ ]:
shutil.copy('/content/drive/MyDrive/Detector_Results/pcmef_corrected_stats.py', '/content/pcmef_corrected_stats.py')
import pcmef_corrected_stats as CS
r_pc = CS.episode_ratio(dfs15, P)   # episode ratio used for the SE correction (15 m/s population)
print('episode ratio (15 m/s):', r_pc)

In [ ]:
# T5c. Corrected paired TOST (margin +-2 pp, same margin as T5). `vals` and `others` come from the T5 cell above.
rows = []
for met in ('f1_macro', 'balanced_accuracy', 'accuracy'):
    for o in others:
        sd = sorted(set(vals['hybrid_full']) & set(vals[o]))
        a, b = (np.array([vals[c][s][met] for s in sd]) * 100 for c in ('hybrid_full', o))
        r = CS.tost_paired_corrected(a, b, margin=2.0, ratio=r_pc)
        rows.append(dict(metric=met, vs=o, diff=r['diff_pp'], ci90_lo=r['ci90_corr'][0], ci90_hi=r['ci90_corr'][1], p=r['p_tost_corr']))
tc = pd.DataFrame(rows)
tc['p_holm'] = tc.groupby('metric')['p'].transform(lambda s: P.holm_bonferroni(s.values))
tc['equivalent_pm2pp'] = tc.p_holm < 0.05
print(tc.round(3).to_string(index=False))
tc.to_csv(f'{OUT_NEXT}/T5c_tost_corrected.csv', index=False)

## BERT cell D at MAX_LEN=288 (15 m/s, 20 seeds)

Macro-F1 (primary) rises at 288 and class 2 starts to be predicted, so cell D is re-run with 20 seeds. The folder name `pilot_bert_maxlen288` is kept for compatibility with existing results.

**Output:** 128 vs 288 table over 20 seeds (accuracy, macro-F1, balanced accuracy, per-class recall), paired differences with CI, Δmodel (D−B) with D@288, and the share of seeds predicting class 2.

**Priority when GPU time is limited:** (1) D@288 at 15 m/s (always runs), (2) D@288 at 25 m/s (`RUN_D25_288`), (3) C@288 at 15 m/s (`RUN_C15_288`; without it the D−C contrast mixes in the MAX_LEN change).

In [ ]:
OUT_PILOT = '/content/drive/MyDrive/Detector_Results/pilot_bert_maxlen288'   # folder name kept for compatibility with existing results; file names include regime and cell
os.makedirs(OUT_PILOT, exist_ok=True)
# Priority when GPU time is limited (run in order, stop any time):
#   (1) BERT D @288, 15 m/s, 20 seeds: always runs
#   (2) BERT D @288, 25 m/s, 20 seeds (RUN_D25_288)
#   (3) BERT C @288, 15 m/s, 20 seeds (RUN_C15_288)
RUN_D25_288 = False
RUN_C15_288 = False
res288 = P.run_factorial(dfs15, 15, 'BERT', OUT_PILOT, seeds=P.SEEDS, cells=[('modified', 'pcmef')], modified_max_len=288)
print('(1) selesai:', len(res288), 'run D 15 m/s')
if RUN_D25_288:
    r_ = P.run_factorial(dfs25, 25, 'BERT', OUT_PILOT, seeds=P.SEEDS, cells=[('modified', 'pcmef')], modified_max_len=288); print('(2) selesai:', len(r_), 'run D 25 m/s')
if RUN_C15_288:
    r_ = P.run_factorial(dfs15, 15, 'BERT', OUT_PILOT, seeds=P.SEEDS, cells=[('modified', 'source_like')], modified_max_len=288); print('(3) selesai:', len(r_), 'run C 15 m/s')

In [ ]:
# 4b.2 Analisis 20 seed: D@128 vs D@288 (berpasangan per seed) dan Δ model (D−B) ulang
from scipy import stats as sps
def L(d, cell, s):
    fp = f'{d}/BERT_15ms_{cell}_seed{s}.json'
    return json.load(open(fp)) if os.path.exists(fp) else None
def per_cls(r):
    cm = np.array(r['confusion_matrix'], float); tp = np.diag(cm)
    return tp / np.maximum(cm.sum(1), 1), cm.sum(0) / cm.sum()
rows = []
for s in P.SEEDS:
    r1, r2, rb = L(OUT, 'D', s), L(OUT_PILOT, 'D', s), L(OUT, 'B', s)
    if r1 is None or r2 is None or rb is None: continue
    rec1, pp1 = per_cls(r1); rec2, pp2 = per_cls(r2)
    rows.append(dict(seed=s, **{f'{k}_128': r1[m]*100 for k, m in (('acc','accuracy'),('f1','f1_macro'),('bal','balanced_accuracy'))},
                     **{f'{k}_288': r2[m]*100 for k, m in (('acc','accuracy'),('f1','f1_macro'),('bal','balanced_accuracy'))},
                     **{f'{k}_B': rb[m]*100 for k, m in (('acc','accuracy'),('f1','f1_macro'),('bal','balanced_accuracy'))},
                     rec2_128=rec1[2]*100, rec2_288=rec2[2]*100, pred2_128=pp1[2]*100, pred2_288=pp2[2]*100))
R = pd.DataFrame(rows); n = len(R); print(f'seed lengkap (D@128, D@288, B): {n}/20')
assert n >= 5
def ci_boot(d, B=10000, seed=0):
    rng = np.random.RandomState(seed); b = np.array([rng.choice(d, len(d), replace=True).mean() for _ in range(B)]); return np.percentile(b, [2.5, 97.5])
print('\n=== D: MAX_LEN 128 vs 288 (selisih = 288 − 128, pp, berpasangan per seed) ===')
out = []
for k, nm in (('f1', 'macro-F1 (primer)'), ('bal', 'balanced accuracy'), ('acc', 'akurasi')):
    d = (R[f'{k}_288'] - R[f'{k}_128']).values; lo, hi = ci_boot(d)
    try: pw = sps.wilcoxon(d)[1]
    except ValueError: pw = np.nan
    out.append(dict(metrik=nm, D_128=R[f'{k}_128'].mean(), D_288=R[f'{k}_288'].mean(), selisih=d.mean(), ci95_lo=lo, ci95_hi=hi,
                    p_t=sps.ttest_1samp(d, 0)[1], p_wilcoxon=pw, d_cohen=d.mean() / d.std(ddof=1)))
print(pd.DataFrame(out).round(3).to_string(index=False))
print('\n=== Kelas 2 (Constant) ===')
print(f"recall kelas 2: 128 = {R.rec2_128.mean():.1f}% | 288 = {R.rec2_288.mean():.1f}%")
print(f"seed yang memprediksi kelas 2 (>0.5% prediksi): 128 -> {(R.pred2_128>0.5).sum()}/{n} | 288 -> {(R.pred2_288>0.5).sum()}/{n}")
print('\n=== Δ model (D − B), berpasangan per seed ===')
dm = []
for k, nm in (('f1', 'macro-F1'), ('bal', 'balanced accuracy'), ('acc', 'akurasi')):
    for L_ in (128, 288):
        d = (R[f'{k}_{L_}'] - R[f'{k}_B']).values; lo, hi = ci_boot(d)
        dm.append(dict(metrik=nm, MAX_LEN=L_, delta_model=d.mean(), ci95_lo=lo, ci95_hi=hi, p_t=sps.ttest_1samp(d, 0)[1]))
dm = pd.DataFrame(dm); print(dm.round(3).to_string(index=False))
for k, nm in (('f1', 'macro-F1'), ('acc', 'akurasi')):
    g128 = dm[(dm.metrik == nm) & (dm.MAX_LEN == 128)].delta_model.iloc[0]; g288 = dm[(dm.metrik == nm) & (dm.MAX_LEN == 288)].delta_model.iloc[0]
    print(f'{nm}: Δ model {g128:+.2f} (128) -> {g288:+.2f} (288); truncation menjelaskan {100*(g288-g128)/abs(g128):.0f}% dari Δ model')
pd.DataFrame(out).to_csv(f'{OUT_NEXT}/L4b_D_128_vs_288.csv', index=False); dm.to_csv(f'{OUT_NEXT}/L4b_delta_model.csv', index=False); R.to_csv(f'{OUT_NEXT}/L4b_per_seed.csv', index=False)

In [ ]:
# 4b.3 Pelengkap (hanya jika dijalankan): D@288 di 25 m/s dan C@288 di 15 m/s
def pair(speed, cell, d288):
    rr = []
    for s in P.SEEDS:
        a, b = L2(OUT, speed, cell, s), L2(d288, speed, cell, s)
        if a is not None and b is not None: rr.append(dict(seed=s, **{f'{k}_128': a[m]*100 for k, m in (('f1','f1_macro'),('bal','balanced_accuracy'),('acc','accuracy'))}, **{f'{k}_288': b[m]*100 for k, m in (('f1','f1_macro'),('bal','balanced_accuracy'),('acc','accuracy'))}))
    return pd.DataFrame(rr)
def L2(d, speed, cell, s):
    fp = f'{d}/BERT_{speed}ms_{cell}_seed{s}.json'; return json.load(open(fp)) if os.path.exists(fp) else None
for speed, cell_, nm in ((25, 'D', 'D 25 m/s'), (15, 'C', 'C 15 m/s')):
    dfp = pair(speed, cell_, OUT_PILOT)
    if len(dfp) < 5: print(f'{nm}: belum cukup seed 288 ({len(dfp)}) -> lewati'); continue
    print(f'\n=== {nm}: 288 vs 128, n={len(dfp)} seed (selisih = 288 - 128, pp) ===')
    for k, m_ in (('f1', 'macro-F1'), ('bal', 'balanced acc'), ('acc', 'akurasi')):
        d = (dfp[f'{k}_288'] - dfp[f'{k}_128']).values; lo, hi = ci_boot(d)
        print(f'{m_:13s} 128={dfp[f"{k}_128"].mean():6.2f}  288={dfp[f"{k}_288"].mean():6.2f}  selisih={d.mean():+.2f}  CI95[{lo:+.2f},{hi:+.2f}]')
# Kontras D-C pada 15 m/s (hanya bila C@288 ada): bandingkan campuran (D288-C128) vs konsisten (D288-C288)
Cd = pair(15, 'C', OUT_PILOT)
if len(Cd) >= 20:
    D288 = {s: L2(OUT_PILOT, 15, 'D', s)['f1_macro']*100 for s in P.SEEDS}; C128 = {s: L2(OUT, 15, 'C', s)['f1_macro']*100 for s in P.SEEDS}; C288 = {s: L2(OUT_PILOT, 15, 'C', s)['f1_macro']*100 for s in P.SEEDS}
    dm_mix = np.array([D288[s] - C128[s] for s in P.SEEDS]); dm_con = np.array([D288[s] - C288[s] for s in P.SEEDS])
    print(f'\nD-C @15 m/s macro-F1: campuran (D288-C128) = {dm_mix.mean():+.2f} pp | konsisten (D288-C288) = {dm_con.mean():+.2f} pp -> pakai yang konsisten untuk suku interaksi')

In [ ]:
# 4b.4 Consistent 2x2 factorial for BERT at 15 m/s: A, B (source, no truncation) + C, D at MAX_LEN=288, compared with the old C, D @128.
def load_cells(cells_dirs, speed=15):
    out = []
    for cell_, d in cells_dirs.items():
        for s in P.SEEDS:
            fp = f'{d}/BERT_{speed}ms_{cell_}_seed{s}.json'
            if os.path.exists(fp): out.append(json.load(open(fp)))
    return out
res_288 = load_cells({'A': OUT, 'B': OUT, 'C': OUT_PILOT, 'D': OUT_PILOT})
res_128 = load_cells({'A': OUT, 'B': OUT, 'C': OUT, 'D': OUT})
fac = {}
for met, nm in (('f1_macro', 'macro-F1'), ('balanced_accuracy', 'balanced accuracy'), ('accuracy', 'accuracy')):
    for tag, rs in (('MAX_LEN=128 (old C,D)', res_128), ('MAX_LEN=288 (consistent C,D)', res_288)):
        sm = P.decompose_factorial(rs, metric=met); fac[(met, tag)] = sm
        P.print_factorial(sm, title=f'BERT 15 m/s | {nm} | {tag}', metric_label=nm)
# Summary: model effect under each protocol (C-A under source-like, D-B under PCMEF) and the interaction
print('\n=== Model effect per protocol (pp) ===')
rows = []
for (met, tag), sm in fac.items():
    c = sm['cells']; rows.append(dict(metric=met, config=tag, A=c['A']['mean'], B=c['B']['mean'], C=c['C']['mean'], D=c['D']['mean'],
                                      C_minus_A=c['C']['mean'] - c['A']['mean'], D_minus_B=c['D']['mean'] - c['B']['mean'],
                                      delta_protokol=sm['delta_protocol']['mean_pp'], interaction=sm['interaction']['mean_pp']))
rs_ = pd.DataFrame(rows); print(rs_.round(2).to_string(index=False)); rs_.to_csv(f'{OUT_NEXT}/L4b_faktorial_128_vs_288.csv', index=False)
print('\nReading: C-A = model effect under the source-like split; D-B = model effect under PCMEF. '
      'If C-A ~ 0 but D-B < 0, the modified model is indistinguishable from the source model under the old split but worse when episodes are separated (model x protocol interaction). '
      'Delta protocol (B-A) does not use C/D and is unchanged.')

In [ ]:
# 4b.5 Corrected factorial decomposition for BERT at 15 m/s (A, B source; C, D at MAX_LEN=288). Uses `res_288` from the cell above.
for met in ('f1_macro', 'balanced_accuracy', 'accuracy'):
    CS.print_corrected(CS.decompose_factorial_corrected(res_288, met, ratio_pcmef=r_pc), f'BERT 15 m/s @288 {met}')

In [ ]:
# T2b. Selisih baseline native vs subset: uji berpasangan per seed (paired t, Wilcoxon, bootstrap) — dasar kalimat "dua populasi dengan catatan"
from scipy import stats as sps
def boot_ci(d, B=10000, seed=0):
    rng = np.random.RandomState(seed); b = np.array([rng.choice(d, len(d), replace=True).mean() for _ in range(B)]); return np.percentile(b, [2.5, 97.5])
rr = []
for m_, g in t2.groupby('model'):
    for k, nm in (('acc', 'akurasi'), ('f1', 'macro-F1')):
        d = (g[f'{k}_subset'] - g[f'{k}_native']).values; lo, hi = boot_ci(d)
        rr.append(dict(model=m_, metrik=nm, native=g[f'{k}_native'].mean(), subset=g[f'{k}_subset'].mean(), selisih=d.mean(), ci95_lo=lo, ci95_hi=hi,
                       p_t=sps.ttest_1samp(d, 0)[1], p_wilcoxon=sps.wilcoxon(d)[1]))
rr = pd.DataFrame(rr); print(rr.round(3).to_string(index=False)); rr.to_csv(f'{OUT_NEXT}/T2b_native_vs_subset_paired.csv', index=False)
print('BiLSTM-15s native predictions are not stored by E4.1; see V1.')

## Subset uniformity, bootstrap TOST check and the 25 m/s regime

The common-key subset is acceptable if (1) the drop on the subset is similar for RF-Stat D, LSTM D and BiLSTM-15s, and (2) no class loses many more windows than the others (checked in T1). V1 trains only the 15 s BiLSTM (20 seeds) to get its native-population predictions. V2 repeats the TOST with 90% bootstrap CIs (non-parametric check). V3 is an optional E4.1 + TOST run at 25 m/s.

In [ ]:
# V1. BiLSTM-15s pada populasi native: native vs subset (model baru; subset diambil dari kunci (episode,t0) yang tersimpan di NPZ E4.1)
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
rowsb = []
for seed in P.SEEDS:
    fp = f'{OUT_E41}/E41N_bilstm15_seed{seed}.npz'
    if not os.path.exists(fp):
        o = P.e41_bilstm15_native(dfs15, seed)
        if o is None: continue
        np.savez_compressed(fp, y_true=o['y_true'], y_pred=o['y_pred'], ep=np.array([k[0] for k in o['keys']], dtype=object), t0=np.array([k[1] for k in o['keys']]))
    z = np.load(fp, allow_pickle=True); zs = np.load(f'{OUT_E41}/E41_15ms_seed{seed}.npz', allow_pickle=True)
    sub = set(zip(zs['ep'].tolist(), np.round(zs['t0'].astype(float), 6)))
    keep = np.array([(e, round(float(t), 6)) in sub for e, t in zip(z['ep'].tolist(), z['t0'])])
    yt, yp = z['y_true'], z['y_pred']
    f = lambda m: (accuracy_score(yt[m], yp[m]) * 100, f1_score(yt[m], yp[m], average='macro', zero_division=0) * 100, balanced_accuracy_score(yt[m], yp[m]) * 100)
    an, fn_, bn = f(np.ones(len(yt), bool)); as_, fs, bs = f(keep)
    rowsb.append(dict(seed=seed, n_native=len(yt), n_subset=int(keep.sum()), acc_native=an, acc_subset=as_, f1_native=fn_, f1_subset=fs, bal_native=bn, bal_subset=bs))
vb = pd.DataFrame(rowsb); print('seed:', len(vb), '| n_subset sama dgn E4.1?', bool((vb.n_subset == [json.load(open(f"{OUT_E41}/E41_15ms_seed{s}.json"))["n_test_common"] for s in vb.seed]).all()))
# Syarat 1: penurunan seragam antar RF-Stat D, LSTM D, BiLSTM-15s (akurasi & macro-F1, rata-rata selisih subset − native)
tab = {'RF_Stat D': t2[t2.model == 'RF_Stat'][['acc_native', 'acc_subset', 'f1_native', 'f1_subset']].mean(),
       'LSTM D': t2[t2.model == 'LSTM'][['acc_native', 'acc_subset', 'f1_native', 'f1_subset']].mean(),
       'BiLSTM-15s': vb[['acc_native', 'acc_subset', 'f1_native', 'f1_subset']].mean()}
sy = pd.DataFrame(tab).T; sy['d_acc'] = sy.acc_subset - sy.acc_native; sy['d_f1'] = sy.f1_subset - sy.f1_native
print(sy.round(2).to_string())
rng_ = sy.d_acc.max() - sy.d_acc.min()
print(f'\nrentang penurunan akurasi antar-detektor = {rng_:.2f} pp (seragam jika <= ~1 pp)')
print('>> Syarat 1 (penurunan seragam):', 'TERPENUHI' if rng_ <= 1.0 and (sy.d_acc < 0).all() else 'TIDAK jelas -> diskusikan / pertimbangkan langkah 1.4')
print('>> Syarat 2 (tidak ada kelas kehilangan jauh lebih banyak): lihat T1 (tersisa 58,5 / 58,9 / 60,1 %)')
vb.to_csv(f'{OUT_NEXT}/V1_bilstm15_native_vs_subset.csv', index=False)

In [ ]:
# V2. TOST: CI 90% bootstrap dari selisih per seed (pemeriksaan non-parametrik untuk T5)
def boot90(d, B=10000, seed=0):
    rng = np.random.RandomState(seed); b = np.array([rng.choice(d, len(d), replace=True).mean() for _ in range(B)]); return np.percentile(b, [5, 95])
rows = []
for met in ('f1_macro', 'balanced_accuracy', 'accuracy'):
    for o in others:
        sd = sorted(set(vals['hybrid_full']) & set(vals[o]))
        d = (np.array([vals['hybrid_full'][s][met] for s in sd]) - np.array([vals[o][s][met] for s in sd])) * 100
        lo, hi = boot90(d); rows.append(dict(metric=met, vs=o, diff_pp=d.mean(), boot90_lo=lo, boot90_hi=hi, di_dalam_pm2=bool(lo > -2 and hi < 2)))
bt = pd.DataFrame(rows); chk = tt.merge(bt, on=['metric', 'vs'], suffixes=('', '_b'))
chk['konsisten'] = chk['ekuivalen_±2pp'] == chk['di_dalam_pm2']
print(chk[['metric', 'vs', 'diff_pp', 'ci90_lo', 'ci90_hi', 'boot90_lo', 'boot90_hi', 'p_holm', 'ekuivalen_±2pp', 'di_dalam_pm2', 'konsisten']].round(3).to_string(index=False))
print('\nBedanya parametrik (T5, Holm) vs bootstrap: pasangan dgn konsisten=False dilaporkan hati-hati.'); chk.to_csv(f'{OUT_NEXT}/V2_tost_bootstrap.csv', index=False)

In [ ]:
# V3. E4.1 + TOST for the 25 m/s regime (optional). All models are near ceiling at 25 m/s, so the result is expected to be trivial.
RUN_25 = False
if RUN_25:
    OUT_E41_25 = '/content/drive/MyDrive/Detector_Results/e41_hybrid_25'; os.makedirs(OUT_E41_25, exist_ok=True)
    res41_25 = P.e41_run_all(dfs25, 25, OUT_E41_25, seeds=P.SEEDS, baselines=('RF_Stat', 'LSTM'))
    R25 = [json.load(open(f)) for f in sorted(glob.glob(f'{OUT_E41_25}/E41_25ms_seed*.json'))]
    v25 = {}
    for r in R25:
        for cfg, mt in r['metrics'].items(): v25.setdefault(cfg, {})[r['seed']] = mt
    for met in ('f1_macro', 'balanced_accuracy', 'accuracy'):
        oth = [c for c in ('RF_Stat_D', 'LSTM_D', 'bilstm_15s', 'xgb_only', 'multi_no_knn', 'multi_no_xgb', 'multi_emb_only') if c in v25]
        r_ = []
        for o in oth:
            sd = sorted(set(v25['hybrid_full']) & set(v25[o]))
            a = np.array([v25['hybrid_full'][s][met] for s in sd]) * 100; b = np.array([v25[o][s][met] for s in sd]) * 100
            r_.append(dict(metric=met, vs=o, hybrid=a.mean(), other=b.mean(), **tost_paired(a, b)))
        t25 = pd.DataFrame(r_); t25['p_holm'] = P.holm_bonferroni(t25.p_tost.values); t25['ekuivalen'] = t25.p_holm < 0.05
        print(t25.round(3).to_string(index=False))
    print('n_test_common 25 m/s: min/median/maks =', min(r['n_test_common'] for r in R25), int(np.median([r['n_test_common'] for r in R25])), max(r['n_test_common'] for r in R25))
else:
    print('RUN_25 = False. Set True to run E4.1 and TOST at 25 m/s (takes about as long as 15 m/s).')

In [ ]:
for model, cells in (('RF_Stat', P.RF_STAT_CELLS), ('LSTM', P.CELLS)):
    for sp, d in ((15, dfs15), (25, dfs25)):
        res = P.run_factorial(d, sp, model, OUT, cells=cells)   # reads cached JSON only
        r = CS.episode_ratio(d, P)
        for met in ('f1_macro', 'balanced_accuracy', 'accuracy'):
            CS.print_corrected(CS.decompose_factorial_corrected(res, met, ratio_pcmef=r), f'{model} {sp} m/s {met}')